This file allows to analyze results obtained by running experiments_competing_risk SYNTHETIC_COMPETING_GROUP.

In [ ]:
import os 
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

import sys

sys.path.append('../')
sys.path.append('../DeepSurvivalMachines/')
from generate import *

In [ ]:
path = 'Results/' # Path where the data is saved

In [ ]:
horizons = [0.25, 0.5, 0.75] # Horizons to evaluate the models
evaluate_groups = True # If True evaluate the group performance

In [ ]:
from NeuralFineGray.experiment import Experiment

In [ ]:
from pycox.evaluation import EvalSurv
from sksurv.metrics import concordance_index_ipcw, brier_score, cumulative_dynamic_auc, integrated_brier_score

def mse(pred, gt):
    # TODO: Check error computation
    return np.sqrt(((pred - gt) ** 2).mean(1)).mean()

### Utils: The evaluatino metrics used
def evaluate(survival_pred, survival_gt, e, t, times, groups = None):
    times_eval = np.quantile(t, horizons)
    results = {}

    train_index = survival_gt.index.difference(survival_pred.index)

    # If multiple risk, compute cause specific metrics
    for r in survival_pred.columns.get_level_values(0).unique():
        res = {}
        e_train, t_train = e[train_index].values, t[train_index].values
        e_test,  t_test  = e[survival_pred.index].values, t[survival_pred.index].values
        g_train, g_test = (None, None) if groups is None else (groups[train_index], groups[survival_pred.index])

        et_train = np.array([(e_train[i] == int(r), t_train[i]) for i in range(len(e_train))], # For estimation censoring
                        dtype = [('e', bool), ('t', float)])
        et_test = np.array([(e_test[i] == int(r), t_test[i]) for i in range(len(e_test))], # For measure performance for given outcome
                        dtype = [('e', bool), ('t', float)])
        
        selection = (t_test < t_train.max()) | (e_test != int(r))
        
        et_test, g_test = et_test[selection], None if groups is None else g_test[selection]
        survival_train = survival_gt.loc[train_index][r]
        survival_fold = survival_pred[r]
        survival_gt_fold = survival_gt.loc[survival_pred.index][r]

        km = EvalSurv(survival_train.T, t_train, e_train == int(r), censor_surv = 'km')
        test_eval = EvalSurv(survival_fold.T, t_test, e_test == int(r), censor_surv = km)

        res['Overall'] = {
                "SE": mse(survival_fold, survival_gt_fold),
                "CIS": test_eval.concordance_td()
            }
        try:
            res['Overall']['BRS'] = test_eval.integrated_brier_score(times.to_numpy())
        except: pass

        for group in groups.unique() if groups is not None else []:
            res['Overall']["SE_{}".format(group)] = mse(survival_fold[selection][g_test == group], survival_gt_fold[selection][g_test == group])
            res['Overall']["Delta_SE_{}".format(group)] = res['Overall']["SE_{}".format(group)] - mse(survival_fold[selection][g_test != group], survival_gt_fold[selection][g_test != group])
        
        if len(times_eval) > 0:
            indexes = [np.argmin(np.abs(times - te)) for te in times_eval]
            briers = brier_score(et_train, et_test, survival_fold[selection].iloc[:, indexes], times_eval)[1]
            for horizon, te, brier, index in zip(horizons, times_eval, briers, indexes):
                try:
                    res[horizon] = {
                        "SE": mse(survival_fold.iloc[:, :index + 1], survival_gt_fold.iloc[:, :index + 1]),
                        "CIS": concordance_index_ipcw(et_train, et_test, 1 - survival_fold[selection].iloc[:, index], te)[0], 
                        "BRS": brier,
                        "ROCS": cumulative_dynamic_auc(et_train, et_test, 1 - survival_fold[selection].iloc[:, index], te)[0][0]}
                except:
                    pass
            
                for group in groups.unique() if groups is not None else []:
                    try:
                        res[horizon]["CIS_{}".format(group)] = concordance_index_ipcw(et_train[g_train == group], et_test[g_test == group], 1 - survival_fold[selection][g_test == group].iloc[:, index], te)[0]
                        res[horizon]["BRS_{}".format(group)] = brier_score(et_train[g_train == group], et_test[g_test == group], survival_fold[selection][g_test == group].iloc[:, index], [horizon])[1][0]
                        res[horizon]["SE_{}".format(group)] = mse(survival_fold[selection][g_test == group].iloc[:, :index + 1], survival_gt_fold[selection][g_test == group].iloc[:, :index + 1])

                        res[horizon]["Delta_CIS_{}".format(group)] = res[horizon]["CIS_{}".format(group)] - concordance_index_ipcw(et_train[g_train != group], et_test[g_test != group], 1 - survival_fold[selection][g_test != group].iloc[:, index], te)[0]
                        res[horizon]["Delta_BRS_{}".format(group)] = res[horizon]["BRS_{}".format(group)] - brier_score(et_train[g_train != group], et_test[g_test != group], survival_fold[selection][g_test != group].iloc[:, index], [te])[1][0]
                        res[horizon]["Delta_SE_{}".format(group)] = res[horizon]["SE_{}".format(group)] - mse(survival_fold[selection][g_test != group].iloc[:, :index + 1], survival_gt_fold[selection][g_test != group].iloc[:, :index + 1])
                    
                    except:
                        pass
        results[r] = pd.DataFrame.from_dict(res)
    results = pd.concat(results)
    results.index.set_names(['Risk', 'Metric'], inplace = True)

    return results

In [ ]:
# Open file and compute performance
predictions, results, rates = {}, {}, {}
for file_name in os.listdir(path):
    if 'generate=' in file_name and '.csv' in file_name: 
        model = file_name
        random_seed = int(model[model.rindex('=') + 1: model.index('_')])
        model = model[model.rindex('_') + 1: model.index('.')]
        print("Opening :", file_name, ' - ', model)

        if model not in results:
            results[model] = {}
            predictions[model] = {}
            rates[model] = {}

        predictions[model][random_seed] = pd.read_csv(path + file_name, header = [0, 1], index_col = 0).dropna().iloc[:, :-1]

        # Remove last columns and change name column to float
        predictions[model][random_seed].columns = pd.MultiIndex.from_frame(pd.DataFrame(index=predictions[model][random_seed].columns).reset_index().astype(float))
        times = predictions[model][random_seed].columns.get_level_values(1).unique()

        # Generate associated ground truth
        x, t, e, betas, outcomes = generate(random_seed)
        cifs = 1 - compute_cif(x, betas, outcomes.cluster, times)
        groups = outcomes.cluster if evaluate_groups else None

        total = outcomes.event.groupby(outcomes.cluster).value_counts()

        # Compute ground truth performance
        rates[model][random_seed] = pd.Series({'Overall': (outcomes.event == 2).mean(),
                                     'Group': total[0][1] / (total[0][1] + total[0][2]) - total[1][1] / (total[1][1] + total[1][2])})

        # Evaluate
        results[model][random_seed] = evaluate(predictions[model][random_seed], cifs, e, t, times, groups)

# Rename
# TODO: Add your method in the list for nicer display
dict_name = {'nfg': 'NeuralFG', 'nfgcs': 'NeuralFG Non Competing', 'finegray': 'Fine Gray', 'dsm': 'DSM', 'dh': 'DeepHit', 'ds': 'DeSurv', 'coxcs': 'CS Cox'} 

results = pd.concat({model: pd.concat(results[model], names = ['Seed']) for model in results}).rename(dict_name)
results.index.set_names('Model', 0, inplace = True)

rates = pd.concat({model: pd.concat(rates[model], names = ['Seed']) for model in rates}).rename(dict_name)
rates.index.set_names('Model', 0, inplace = True)

In [ ]:
# Compute average performance across fold and models
table = results.groupby(['Model', 'Risk', 'Metric']).apply(lambda x:  pd.Series(["{:.3f} ({:.3f})".format(mean, std) for mean, std in zip(x.mean(), x.std())], index = x.columns))
table = table.unstack(level=-1).stack(level=0).unstack(level=-1).loc[:, ['Delta_SE_0']]
table = table.reorder_levels(['Risk', 'Model']).sort_index(level = 0, sort_remaining = False)

table

### Display bias error given theoretical error

In [ ]:
# Matched between competing and non competing
matched = {
    'NeuralFG': 'NeuralFG Non Competing'
}

In [ ]:
for key, value in matched.items():
    for seed in results.index.get_level_values('Seed').unique():
        error_C = results.loc[(key, seed, 1.0, 'SE')].Overall
        error_NC = results.loc[(value, seed, 1.0, 'SE')].Overall
        predicted_error = rates.loc[(key, seed, 'Overall')]

        plt.scatter(error_C - error_NC, predicted_error)

plt.xlabel('Observed gain')
plt.ylabel('Predicted gain')

In [ ]:
for key, value in matched.items():
    for seed in results.index.get_level_values('Seed').unique():
        gain_C = results.loc[(key, seed, 1.0, 'Delta_SE_0')].Overall
        gain_NC = results.loc[(value, seed, 1.0, 'Delta_SE_0')].Overall
        predicted_gain = rates.loc[(key, seed, 'Group')]

        plt.scatter(gain_NC - gain_C, predicted_gain)

plt.xlabel('Observed group gain')
plt.ylabel('Predicted group gain')